In [0]:
# Configuração da chave de acesso
spark.conf.set(
    "fs.azure.account.key.stprojeto2zaca.dfs.core.windows.net",
    "7kEJbeQdNyThcjCw9jHUrHBG4nCsf67Oeltzaw/5pW7ptO3bhOM2s5cZ7comiXjgJHd04GMim77r+ASt/wB8Ow=="
)

In [0]:
# 1. Definir o caminho do arquivo na camada Bronze
path_bronze = "abfss://dados@stprojeto2zaca.dfs.core.windows.net/01-bronze/train_sample.csv"

# 2. Ler o arquivo CSV usando Spark
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_bronze)

# 3. Mostrar os primeiros resultados
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Limpeza simples: removendo corridas com valor zero ou negativo
df_silver = df.filter(col("fare_amount") > 0) \
              .withColumn("data_processamento", current_timestamp())

# 2. Definir o caminho de destino na Silver
path_silver = "abfss://dados@stprojeto2zaca.dfs.core.windows.net/02-silver/tabela_corridas"

# 3. Salvar em formato DELTA (o mais potente do mercado)
df_silver.write.format("delta") \
         .mode("overwrite") \
         .save(path_silver)

print("Dados salvos com sucesso na camada Silver!")

In [0]:
from pyspark.sql.functions import avg, round

# 1. Ler o dado que você acabou de salvar na Silver
df_silver_ready = spark.read.format("delta").load(path_silver)

# 2. Criar o insight: Média de preço por grupo de passageiros
df_gold = df_silver_ready.groupBy("passenger_count") \
                         .agg(round(avg("fare_amount"), 2).alias("media_valor_corrida")) \
                         .orderBy("passenger_count")

# 3. Mostrar o insight final
display(df_gold)

# 4. Salvar o resultado final na camada Gold para o Power BI ou analistas
path_gold = "abfss://dados@stprojeto2zaca.dfs.core.windows.net/03-gold/insight_passageiros"
df_gold.write.format("delta").mode("overwrite").save(path_gold)

print("Insight gerado e salvo na camada Gold com sucesso!")